# SECOM Semiconductor Manufacturing: Which Process Signals Relate to Product Failure?

## Business Problem
A fab records 590 sensor measurements for each production sample, and each sample is
later inspected as pass or fail. About 6.6% of samples fail. This project asks:
1. Which sensor signals are most associated with failures?
2. Do failure rates and sensor readings drift over time?
3. Can sensor data flag high-risk samples before inspection, and how reliable is such a model?

## Data
UCI SECOM dataset: 1,567 samples, 590 sensors, about 3 months (Jul to Oct 2008).
License: CC BY 4.0.

## Approach
Data quality checks -> time trends -> key signal screening -> baseline model -> business recommendations and limitations

In [2]:
import pandas as pd

X = pd.read_csv("../data/raw/secom.data", sep=r"\s+", header=None, na_values="NaN")
X.columns = [f"sensor_{i}" for i in range(X.shape[1])]

y = pd.read_csv("../data/raw/secom_labels.data", sep=" ", header=None,
                names=["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"].str.strip('"'),
                                format="%d/%m/%Y %H:%M:%S")

df = pd.concat([X, y], axis=1)
print(df.shape)
print(df["label"].value_counts())
df.head()

(1567, 592)
label
-1    1463
 1     104
Name: count, dtype: int64


,sensor_0,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,...,sensor_582,sensor_583,sensor_584,sensor_585,sensor_586,sensor_587,sensor_588,sensor_589,label,timestamp
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1,2008-07-19 11:55:00
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1,2008-07-19 12:32:00
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,2008-07-19 13:17:00
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1,2008-07-19 14:43:00
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1,2008-07-19 15:22:00


In [4]:
# shape
print(df.shape)

# fail rate
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))

# timestamp range
print(df["timestamp"].min(), df["timestamp"].max())

# missing values
missing = df.drop(columns=["label", "timestamp"]).isna().mean().sort_values(ascending=False)
print((missing == 0).sum(), "columns no missing")
print(missing.head(10).round(3))

# constant
sensors = df.drop(columns=["label", "timestamp"])
print((sensors.nunique() <= 1).sum(), "constant columns")

(1567, 592)
label
-1    1463
 1     104
Name: count, dtype: int64
label
-1    0.934
 1    0.066
Name: proportion, dtype: float64
2008-07-19 11:55:00 2008-10-17 06:07:00
52 columns no missing
sensor_157    0.912
sensor_292    0.912
sensor_293    0.912
sensor_158    0.912
sensor_492    0.856
sensor_358    0.856
sensor_85     0.856
sensor_220    0.856
sensor_246    0.650
sensor_109    0.650
dtype: float64
116 constant columns


In [6]:
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

# missing rates: find how many sensors exceed 5%, 20%, 50% missing
missing = df[sensor_cols].isna().mean()
for t in [0.05, 0.2, 0.5]:
    print(f"Sensors with more than {t:.0%} missing: {(missing > t).sum()}")

# constant sensors
constant_cols = [c for c in sensor_cols if df[c].nunique() <= 1]
print("Number of constant sensors:", len(constant_cols))

# check whether constant sensors have any missing values
print("Average missing rate of constant sensors:", df[constant_cols].isna().mean().mean().round(3))

Sensors with more than 5% missing: 52
Sensors with more than 20% missing: 32
Sensors with more than 50% missing: 28
Number of constant sensors: 116
Average missing rate of constant sensors: 0.004


## Data cleaning decisions
1. Dropped 116 constant sensors: a sensor with one unique value cannot separate pass from fail.
2. Dropped 28 sensors with more than 50% missing values.
3. I did not impute the remaining missing values yet. I will split the data into training
and test sets first, and calculate the fill values from the training set only.
This keeps the test set unseen and avoids data leakage.

In [8]:
# drop constant sensors and sensors with more than 50% missing
high_missing_cols = missing[missing > 0.5].index.tolist()
cols_to_drop = sorted(set(constant_cols) | set(high_missing_cols))

print("Constant sensors:", len(constant_cols))
print("High-missing sensors (>50%):", len(high_missing_cols))
print("Overlap between the two groups:", len(set(constant_cols) & set(high_missing_cols)))
print("Total sensors dropped:", len(cols_to_drop))

# build the cleaned df
df_clean = df.drop(columns=cols_to_drop)
remaining_sensors = [c for c in df_clean.columns if c.startswith("sensor_")]
print("Remaining sensors:", len(remaining_sensors))
print("Shape of df_clean:", df_clean.shape)

# remaining missing values (handle later...)
remaining_missing = df_clean[remaining_sensors].isna().mean()
print("Remaining sensors with any missing values:", (remaining_missing > 0).sum())
print("Largest remaining missing rate:", remaining_missing.max().round(3))

Constant sensors: 116
High-missing sensors (>50%): 28
Overlap between the two groups: 0
Total sensors dropped: 144
Remaining sensors: 446
Shape of df_clean: (1567, 448)
Remaining sensors with any missing values: 394
Largest remaining missing rate: 0.456


In [9]:
# number of missing sensor values per sample (in row), compared between pass and fail
df_clean = df_clean.copy()
df_clean["n_missing"] = df_clean[remaining_sensors].isna().sum(axis=1)
print(df_clean.groupby("label")["n_missing"].describe().round(1))

# per-sensor missing rate for pass vs fail, and the sensors with the largest gap
miss_by_label = df_clean.groupby("label")[remaining_sensors].apply(lambda g: g.isna().mean()).T
miss_by_label.columns = ["pass_missing_rate", "fail_missing_rate"]
miss_by_label["gap"] = (miss_by_label["fail_missing_rate"] - miss_by_label["pass_missing_rate"]).abs()
print(miss_by_label.sort_values("gap", ascending=False).head(10).round(3))

        count  mean  std  min  25%  50%   75%    max
label                                               
-1     1463.0   7.0  9.7  0.0  0.0  4.0  12.0  103.0
 1      104.0   5.9  7.4  0.0  0.0  4.0   8.0   44.0
            pass_missing_rate  fail_missing_rate    gap
sensor_385              0.466              0.317  0.149
sensor_112              0.466              0.317  0.149
sensor_519              0.466              0.317  0.149
sensor_247              0.466              0.317  0.149
sensor_567              0.170              0.231  0.061
sensor_563              0.170              0.231  0.061
sensor_562              0.170              0.231  0.061
sensor_565              0.170              0.231  0.061
sensor_566              0.170              0.231  0.061
sensor_564              0.170              0.231  0.061


In [7]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
df_clean.drop(columns=["n_missing"]).to_csv("../data/processed/secom_clean.csv", index=False)
print("Saved.")

Saved.
